# **24IT105 Practical - 2**

## **Problem Definition:**  
### Programmatically interface with diverse file-based, API-driven, and relational source database environments. Students will write generation engines to simulate transactional workloads, profile incoming files to automatically discover schemas, isolate data quality violations (such as null-entry leaks and format divergence), and establish appropriate operational ingestion criteria.

**STEP 1 : Install Required Library**

In [1]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.8 MB/s eta 0:00:00


**STEP 2 : Import Required Libraries**

In [ ]:
import pandas as pd
import random
import sqlite3
import json
import requests
import re

from faker import Faker

: 

**STEP 3 – Initialize Faker**

In [3]:
fake = Faker()

print("Faker Initialized Successfully")

Faker Initialized Successfully


**STEP 4 – Generate 10,000 Customer Records**

In [4]:
customers = []

for i in range(1, 10001):

    customer = {

        "CustomerID": i,

        "Name": fake.name(),

        "Email": fake.email(),

        "Phone": fake.msisdn()[:10],

        "Age": random.randint(18,60),

        "City": fake.city()

    }

    customers.append(customer)

df = pd.DataFrame(customers)

display(df.head())

,CustomerID,Name,Email,Phone,Age,City
0,1,Kathy Liu,rlewis@example.com,7479367846,19,North Latoya
1,2,Troy Hickman,annjohnson@example.com,1724700870,32,South Brittany
2,3,Dawn Parker,matthew38@example.com,7784717911,30,Changfurt
3,4,Emily Lewis,brittanyfisher@example.net,7364177897,47,New Debraton
4,5,Aaron Welch,turnerchase@example.com,4230409361,26,Brownfurt


**STEP 5 – Verify Total Records**

In [5]:
print("Total Records :", len(df))

Total Records : 10000


**STEP 6 – Inject 1000 Invalid Records**

In [6]:
invalid_rows = random.sample(range(10000),1000)

for row in invalid_rows:

    error_type = random.randint(1,7)

    if error_type == 1:
        df.loc[row,"Name"] = None

    elif error_type == 2:
        df.loc[row,"Email"] = None

    elif error_type == 3:
        df.loc[row,"Email"] = "abcgmail.com"

    elif error_type == 4:
        df.loc[row,"Phone"] = "123"

    elif error_type == 5:
        df.loc[row,"Age"] = 150

    elif error_type == 6:
        df.loc[row,"City"] = None

    elif error_type == 7:
        df.loc[row,"CustomerID"] = 500

**STEP 7 – Save Dataset**

In [7]:
df.to_csv("customer_profiles.csv",index=False)

print("CSV Saved Successfully")

CSV Saved Successfully


**STEP 8 – Read CSV**

In [8]:
df = pd.read_csv("customer_profiles.csv")

display(df.head())

,CustomerID,Name,Email,Phone,Age,City
0,1,Kathy Liu,rlewis@example.com,7479367846,19,North Latoya
1,2,Troy Hickman,annjohnson@example.com,1724700870,32,South Brittany
2,3,Dawn Parker,matthew38@example.com,7784717911,30,Changfurt
3,4,Emily Lewis,brittanyfisher@example.net,7364177897,47,New Debraton
4,5,Aaron Welch,turnerchase@example.com,4230409361,26,Brownfurt


**STEP 9 – Schema Discovery**

In [9]:
print("Shape")

print(df.shape)

print()

print("Columns")

print(df.columns)

print()

print("Data Types")

print(df.dtypes)

Shape
(10000, 6)

Columns
Index(['CustomerID', 'Name', 'Email', 'Phone', 'Age', 'City'], dtype='object')

Data Types
CustomerID     int64
Name          object
Email         object
Phone          int64
Age            int64
City          object
dtype: object


**STEP 10 – Dataset Information**

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   CustomerID  10000 non-null  int64 
 1   Name        9854 non-null   object
 2   Email       9857 non-null   object
 3   Phone       10000 non-null  int64 
 4   Age         10000 non-null  int64 
 5   City        9864 non-null   object
dtypes: int64(3), object(3)
memory usage: 468.9+ KB


**STEP 11 – Summary Statistics**

In [11]:
df.describe(include="all")

,CustomerID,Name,Email,Phone,Age,City
count,10000.000000,9854,9857,1.000000e+04,10000.000000,9864
unique,NaN,9264,9515,NaN,NaN,7603
top,NaN,Brian Williams,abcgmail.com,NaN,NaN,Port Michael
freq,NaN,5,147,NaN,NaN,11
mean,4943.476700,NaN,NaN,4.919006e+09,40.888500,NaN
std,2911.147752,NaN,NaN,2.931506e+09,18.210234,NaN
min,1.000000,NaN,NaN,1.230000e+02,18.000000,NaN
25%,2403.750000,NaN,NaN,2.341164e+09,29.000000,NaN
50%,4936.500000,NaN,NaN,4.915414e+09,40.000000,NaN
75%,7465.250000,NaN,NaN,7.460328e+09,50.000000,NaN


**STEP 12 – Find Missing Values**

In [12]:
print(df.isnull().sum())

CustomerID      0
Name          146
Email         143
Phone           0
Age             0
City          136
dtype: int64


**STEP 13 – Find Duplicate Records**

In [13]:
print("Duplicate Rows :",df.duplicated().sum())

Duplicate Rows : 0


**STEP 14 – Invalid Email Detection**

In [14]:
invalid_email = df[
    ~df["Email"].fillna("").str.contains(
        r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$',
        regex=True
    )
]

print("Invalid Emails")

print(invalid_email[["CustomerID","Email"]].head())

print()

print("Total Invalid Emails :",len(invalid_email))

Invalid Emails
     CustomerID         Email
53           54           NaN
55           56  abcgmail.com
142         143           NaN
198         199  abcgmail.com
226         227  abcgmail.com

Total Invalid Emails : 290


**STEP 15 – Invalid Phone Numbers**

In [15]:
invalid_phone = df[
    ~df["Phone"].astype(str).str.fullmatch(r"\d{10}")
]

print("Invalid Phone Numbers")

print(invalid_phone[["CustomerID","Phone"]].head())

print()

print("Total Invalid Phones :",len(invalid_phone))

Invalid Phone Numbers
    CustomerID      Phone
6            7  716227983
9           10        123
16          17  102105154
21          22  829385870
28          29  296292266

Total Invalid Phones : 1108


**STEP 16 – Invalid Age**

In [16]:
invalid_age = df[
    (df["Age"]<18) | (df["Age"]>60)
]

print("Invalid Age Records")

display(invalid_age.head())

print()

print("Total Invalid Ages :",len(invalid_age))

Invalid Age Records


,CustomerID,Name,Email,Phone,Age,City
71,72,Tara Best,austin53@example.com,3059684833,150,Brownchester
231,232,Brett Murphy,jeremyreed@example.com,8293789679,150,Dixonbury
235,236,Ashley Lamb,mark02@example.net,4231118065,150,East Chadport
277,278,Deborah Simmons,franklinpatterson@example.org,8833467402,150,Williamport
278,279,Kyle Carpenter,robinschultz@example.net,6680020605,150,Lake Williamchester



Total Invalid Ages : 149


**STEP 17 – Create Valid and Invalid Data**

In [17]:
# Email validation
email_pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

# Phone validation
phone_pattern = r'^\d{10}$'

# Create validation conditions
valid_condition = (
    df["Name"].notna() &
    df["Email"].fillna("").str.match(email_pattern) &
    df["Phone"].astype(str).str.match(phone_pattern) &
    df["Age"].between(18, 60) &
    df["City"].notna() &
    (~df["CustomerID"].duplicated())
)

# Separate datasets
valid_df = df[valid_condition].copy()
invalid_df = df[~valid_condition].copy()

print("Valid Records :", len(valid_df))
print("Invalid Records :", len(invalid_df))

Valid Records : 8138
Invalid Records : 1862


**STEP 18 – Save Valid and Invalid Records**

In [18]:
valid_df.to_csv("valid_customers.csv", index=False)
invalid_df.to_csv("invalid_customers.csv", index=False)

print("Files Created Successfully")

Files Created Successfully


**STEP 19 – Display Sample Records**

In [19]:
print("Valid Customers")
display(valid_df.head())

print("\nInvalid Customers")
display(invalid_df.head())

Valid Customers


,CustomerID,Name,Email,Phone,Age,City
0,1,Kathy Liu,rlewis@example.com,7479367846,19,North Latoya
1,2,Troy Hickman,annjohnson@example.com,1724700870,32,South Brittany
2,3,Dawn Parker,matthew38@example.com,7784717911,30,Changfurt
3,4,Emily Lewis,brittanyfisher@example.net,7364177897,47,New Debraton
4,5,Aaron Welch,turnerchase@example.com,4230409361,26,Brownfurt



Invalid Customers


,CustomerID,Name,Email,Phone,Age,City
6,7,Rita Wilson MD,kevinbright@example.net,716227983,22,Jacquelineborough
9,10,Sarah Thornton,rrios@example.com,123,25,Lake Joseph
13,14,Ralph Burton,washingtonrobert@example.org,3748349263,37,NaN
16,17,Scott Lewis,stephaniebender@example.net,102105154,29,Fergusontown
21,22,Andrew Davis,wallermichael@example.com,829385870,18,Smithborough


**STEP 20 – Display Final Summary**

In [20]:
print("="*40)
print("DATA QUALITY SUMMARY")
print("="*40)

print("Total Records :", len(df))
print("Valid Records :", len(valid_df))
print("Invalid Records :", len(invalid_df))

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Customer IDs :", df["CustomerID"].duplicated().sum())

print("="*40)

DATA QUALITY SUMMARY
Total Records : 10000
Valid Records : 8138
Invalid Records : 1862

Missing Values
CustomerID      0
Name          146
Email         143
Phone           0
Age             0
City          136
dtype: int64

Duplicate Customer IDs : 121


**STEP 21 – Fetch API Data**

In [21]:
url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(url)

api_data = response.json()

print("API Records Downloaded :", len(api_data))

API Records Downloaded : 10


**STEP 22 – Save API Data**

In [22]:
with open("transactions.json", "w") as file:
    json.dump(api_data, file, indent=4)

print("transactions.json Created")

transactions.json Created


**STEP 23 – Read API JSON**

In [23]:
with open("transactions.json", "r") as file:
    data = json.load(file)

print(data[0])

{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}


**STEP 24 – Extract Nested JSON Data**

In [24]:
api_df = pd.json_normalize(data)

display(api_df.head())

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


**STEP 25 – Select Required Columns**

In [25]:
api_df = api_df[
    [
        "id",
        "name",
        "username",
        "email",
        "address.city",
        "company.name"
    ]
]

display(api_df.head())

,id,name,username,email,address.city,company.name
0,1,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,South Elvis,Robel-Corkery
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC


**STEP 26 – Create Configuration File**

In [26]:
config = """
Server=localhost
Database=CustomerDB
Port=5432
Username=admin
Timeout=60
Logging=True
Retry=3
"""

with open("config.txt", "w") as file:
    file.write(config)

print("config.txt Created")

config.txt Created


**STEP 27 – Read Configuration File**

In [27]:
config_data = {}

with open("config.txt", "r") as file:

    for line in file:

        if "=" in line:

            key, value = line.strip().split("=")

            config_data[key] = value

print(config_data)

{'Server': 'localhost', 'Database': 'CustomerDB', 'Port': '5432', 'Username': 'admin', 'Timeout': '60', 'Logging': 'True', 'Retry': '3'}


**STEP 28 – Create SQLite Database**

In [28]:
conn = sqlite3.connect("customer.db")

print("Database Created")

Database Created


**STEP 29 – Load Valid Customers into Database**

In [29]:
valid_df.to_sql(
    "Customers",
    conn,
    if_exists="replace",
    index=False
)

print("Customers Loaded Successfully")

Customers Loaded Successfully


**STEP 30 – Verify Database Records**

In [30]:
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM Customers")

print("Total Records in Database :", cursor.fetchone()[0])

Total Records in Database : 8138


**STEP 31 – Display First 10 Records from Database**

In [31]:
query = "SELECT * FROM Customers LIMIT 10"

database_df = pd.read_sql(query, conn)

display(database_df)

,CustomerID,Name,Email,Phone,Age,City
0,1,Kathy Liu,rlewis@example.com,7479367846,19,North Latoya
1,2,Troy Hickman,annjohnson@example.com,1724700870,32,South Brittany
2,3,Dawn Parker,matthew38@example.com,7784717911,30,Changfurt
3,4,Emily Lewis,brittanyfisher@example.net,7364177897,47,New Debraton
4,5,Aaron Welch,turnerchase@example.com,4230409361,26,Brownfurt
5,6,Dominique Gray,zacharynguyen@example.net,7217470497,32,Allisonstad
6,8,Susan Gray,whitemaria@example.com,3739845867,37,Swansonmouth
7,9,Diane Adams,ojones@example.org,4971995173,31,Masseyside
8,11,Jocelyn Gonzalez,boyddenise@example.net,3855424787,21,Collierville
9,12,Annette Noble,graykayla@example.com,8896275677,52,Justinmouth


**STEP 32 – Close Database**

In [32]:
conn.close()

print("Database Closed")

Database Closed


**STEP 33 – Download All Output Files (Google Colab)**

In [33]:
from google.colab import files

**STEP 34 – Download Customer Dataset**

In [34]:
files.download("customer_profiles.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**STEP 35 – Download Valid Dataset**

In [35]:
files.download("valid_customers.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**STEP 36 – Download Invalid Dataset**

In [36]:
files.download("invalid_customers.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**STEP 37 – Download API JSON**

In [37]:
files.download("transactions.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**STEP 38 – Download SQLite Database**

In [38]:
files.download("customer.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>